# Standard pH Calibration  (pH 1 - 12)

Calibrate a glass pH electrode and convert measured EMF into pH.
The response is assumed **linear (Nernstian)**:

$$\text{EMF} = E_0 - S\cdot \text{pH}$$

Choose one of two options:

- **Option 1 - pH at a SPECIFIC temperature.** Calibrate with buffers measured at one
  temperature, then predict sample pH at that same temperature.
- **Option 2 - pH at ANY temperature.** Calibrate with buffers measured at several
  temperatures; the notebook fits $E_0(T)$ and $S(T)$ so you can predict pH at any
  temperature in the calibrated range.

All input is via **inline arrays** - just edit the values in the marked cells and run.

*From Saleesongsom et al., ACS Omega (2026).*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats

np.set_printoptions(suppress=True)


---
## Option 1 - pH at a specific temperature

### 1a. Build the calibration
Enter buffer pH / EMF pairs measured at a single temperature (>= 2 points, 3+ better).


In [ ]:
# ===== EDIT YOUR NIST/DIN BUFFER DATA (single temperature) =====
# Standard pH calibration uses NIST/DIN buffers at pH 4, 7 and 10.
buffer_pH  = [4.009, 7.012, 10.015]   # NIST/DIN buffer pH values
buffer_EMF = [181.5, 5.9, -170.9]     # measured EMF (mV)
T_C        = 24.4                      # measurement temperature (deg C)
# ===============================================================

buffer_pH  = np.asarray(buffer_pH,  float)
buffer_EMF = np.asarray(buffer_EMF, float)

slope, intercept, r, p, se = stats.linregress(buffer_pH, buffer_EMF)
S1, E0_1, r2_1 = -slope, intercept, r**2
S_theory = 2.302585 * 8.314462 * (T_C + 273.15) / 96485.0 * 1000.0  # mV/pH

print(f"Calibration @ {T_C} C:  EMF = {E0_1:.2f} - {S1:.3f} * pH")
print(f"  slope S = {S1:.3f} mV/pH   E0 = {E0_1:.2f} mV   R^2 = {r2_1:.5f}")
print(f"  theoretical Nernst slope = {S_theory:.3f} mV/pH  ({100*S1/S_theory:.1f}% Nernstian)")

fig, ax = plt.subplots(figsize=(4, 3.2), dpi=120)
xl = np.array([0, 14])
ax.plot(xl, E0_1 - S1 * xl, "-", color="#E8261C", lw=1.6,
        label=f"EMF = {E0_1:.1f} - {S1:.2f}·pH\n$R^2$ = {r2_1:.4f}")
ax.scatter(buffer_pH, buffer_EMF, s=70, c="#F5A623", edgecolors="black", zorder=3)
ax.set_xlim(0, 14); ax.set_xlabel("pH"); ax.set_ylabel("EMF (mV)")
ax.legend(fontsize=8, frameon=False); ax.set_title(f"Standard calibration @ {T_C} °C")
plt.tight_layout(); plt.show()


### 1b. Predict sample pH
Enter the EMF of your sample(s) measured at the same temperature.
$$\text{pH} = (E_0 - \text{EMF})/S$$


In [ ]:
# ============ EDIT YOUR SAMPLE EMF (same temperature) ============
sample_EMF = [120.0, 30.0, -60.0]     # measured EMF (mV)
# =================================================================

sample_EMF = np.atleast_1d(np.asarray(sample_EMF, float))
sample_pH  = (E0_1 - sample_EMF) / S1

print(f"Predicted pH @ {T_C} C:")
for e, ph in zip(sample_EMF, sample_pH):
    print(f"  EMF {e:8.2f} mV  ->  pH {ph:6.3f}")


---
## Option 2 - pH at any temperature

### 2a. Build the temperature-dependent calibration
Enter buffer pH / EMF curves measured at **several temperatures** (>= 2, 3+ recommended).
The notebook fits a line at each temperature, then models how $E_0$ and $S$ vary with $T$:

$$E_0(T)\ \text{and}\ S(T)\ \text{as linear functions of }T,\qquad
  \text{EMF}(\text{pH},T)=E_0(T)-S(T)\cdot\text{pH}$$


In [ ]:
# ============ EDIT YOUR BUFFER DATA (multiple temperatures) ============
# NIST/DIN buffers at pH 4, 7, 10 measured at each temperature.
# Format:  temperature_C : ( [buffer pH], [measured EMF in mV] )
calibration_data = {
    15.0: ([4.01, 7.01, 10.01], [179.8,  9.4, -161.0]),
    25.0: ([4.01, 7.01, 10.01], [183.1,  6.7, -169.7]),
    40.0: ([4.01, 7.01, 10.01], [188.1,  2.7, -182.5]),
}
# ======================================================================

temps = np.array(sorted(calibration_data), float)
E0_list, S_list = [], []
for T in temps:
    ph, emf = map(lambda a: np.asarray(a, float), calibration_data[T])
    sl, ic, r, *_ = stats.linregress(ph, emf)
    E0_list.append(ic); S_list.append(-sl)
    print(f"T = {T:5.1f} C   E0 = {ic:7.2f} mV   S = {-sl:6.3f} mV/pH   R^2 = {r**2:.5f}")
E0_arr, S_arr = np.asarray(E0_list), np.asarray(S_list)

# linear temperature models
cE = np.polyfit(temps, E0_arr, 1)
cS = np.polyfit(temps, S_arr, 1)
E0_of_T = lambda T: np.polyval(cE, T)
S_of_T  = lambda T: np.polyval(cS, T)
print(f"\nE0(T) = {cE[0]:+.4f}*T + {cE[1]:.3f}")
print(f"S(T)  = {cS[0]:+.4f}*T + {cS[1]:.3f}")


In [ ]:
# ---- Plots: E0(T), S(T), and the EMF(pH,T) family ----
Tg = np.linspace(temps.min(), temps.max(), 100)
fig, axs = plt.subplots(1, 3, figsize=(11, 3), dpi=120)
axs[0].scatter(temps, E0_arr, c="#E8261C", edgecolors="k", zorder=3); axs[0].plot(Tg, E0_of_T(Tg), "k-")
axs[0].set_xlabel("T (°C)"); axs[0].set_ylabel("$E_0$ (mV)"); axs[0].set_title("$E_0(T)$")
axs[1].scatter(temps, S_arr, c="#E8261C", edgecolors="k", zorder=3); axs[1].plot(Tg, S_of_T(Tg), "k-")
axs[1].set_xlabel("T (°C)"); axs[1].set_ylabel("S (mV/pH)"); axs[1].set_title("$S(T)$")

pH_grid = np.linspace(1, 12, 200)
cmap = plt.cm.autumn_r; norm = mpl.colors.Normalize(temps.min(), temps.max())
for T in np.linspace(temps.min(), temps.max(), 40):
    axs[2].plot(pH_grid, E0_of_T(T) - S_of_T(T) * pH_grid, color=cmap(norm(T)), lw=0.9)
axs[2].set_xlim(1, 12); axs[2].set_xlabel("pH"); axs[2].set_ylabel("EMF (mV)")
axs[2].set_title("EMF(pH, T)")
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=axs[2], pad=0.02).set_label("T (°C)")
plt.tight_layout(); plt.show()


### 2b. Predict sample pH at any temperature
Enter each sample EMF **and** its temperature.
$$\text{pH} = \dfrac{E_0(T)-\text{EMF}}{S(T)}$$


In [ ]:
# ============ EDIT YOUR SAMPLE MEASUREMENTS ============
sample_EMF = [150.0, 0.0, -150.0]     # measured EMF (mV)
sample_T   = [20.0, 30.0, 35.0]       # temperature of each sample (deg C)
# ======================================================

sample_EMF = np.atleast_1d(np.asarray(sample_EMF, float))
sample_T   = np.atleast_1d(np.asarray(sample_T, float))
if sample_T.size == 1:
    sample_T = np.full_like(sample_EMF, sample_T[0])

print("EMF (mV) @ T (C)  ->  predicted pH")
for e, T in zip(sample_EMF, sample_T):
    ph = (E0_of_T(T) - e) / S_of_T(T)
    warn = "  (T outside calibrated range)" if (T < temps.min() or T > temps.max()) else ""
    print(f"  {e:8.2f} @ {T:5.1f}  ->  pH {ph:6.3f}{warn}")
